# NFL Upset Prediction: LSTM Model

This notebook trains and evaluates the LSTM model with attention:
1. Prepare sequence data for LSTM
2. Train LSTM with PyTorch
3. Attention weight analysis
4. Hyperparameter tuning
5. Save trained model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss

# Project imports
import sys
sys.path.insert(0, '..')

from src.models.lstm_model import UpsetLSTM
from src.models.cv_splitter import TimeSeriesCVSplitter
from src.evaluation.metrics import EvaluationMetrics

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load and Prepare Sequence Data

In [ ]:
# Load engineered features
df = pd.read_parquet('../data/processed/features_engineered.parquet')
print(f"Loaded {len(df):,} records")

# Load feature columns
with open('../data/processed/feature_columns.txt', 'r') as f:
    feature_columns = [line.strip() for line in f.readlines()]
print(f"Feature columns: {len(feature_columns)}")

In [ ]:
# Separate sequence features (rolling) from matchup features (static)
rolling_features = [c for c in feature_columns if c.startswith('rolling_')]
matchup_features = [c for c in feature_columns if c.startswith(('matchup_', 'spread'))]

print(f"Rolling features (sequence): {len(rolling_features)}")
print(f"Matchup features (static): {len(matchup_features)}")

In [ ]:
# Prepare data - drop rows with missing values
all_features = rolling_features + matchup_features
df_clean = df.dropna(subset=all_features + ['is_upset'])
print(f"Records after dropping missing: {len(df_clean):,}")

X_rolling = df_clean[rolling_features].values
X_matchup = df_clean[matchup_features].values
y = df_clean['is_upset'].values.astype(np.float32)

print(f"\nRolling features shape: {X_rolling.shape}")
print(f"Matchup features shape: {X_matchup.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# LSTM expects sequences - reshape rolling features
# For now, treat the rolling features as a single time step
# In a more advanced version, we could create actual game sequences
SEQUENCE_LENGTH = 5  # Represents 5-game rolling window conceptually

# Reshape for LSTM: (batch, seq_len, features)
# We'll repeat the rolling features to simulate a sequence
n_samples = len(X_rolling)
n_rolling_features = len(rolling_features)
n_matchup_features = len(matchup_features)

# Create sequence data by reshaping
X_seq = X_rolling.reshape(n_samples, 1, n_rolling_features)  # (N, 1, features)
X_seq = np.repeat(X_seq, SEQUENCE_LENGTH, axis=1)  # (N, seq_len, features)

print(f"Sequence input shape: {X_seq.shape}")
print(f"Matchup input shape: {X_matchup.shape}")

## 2. Setup Time-Series CV and Training

In [ ]:
# Standardize features
scaler_seq = StandardScaler()
scaler_matchup = StandardScaler()

# Time-series CV splitter
cv_splitter = TimeSeriesCVSplitter(n_folds=3)

In [ ]:
def train_lstm(model, X_seq_train, X_matchup_train, y_train, 
               X_seq_val, X_matchup_val, y_val, 
               epochs=50, batch_size=64, lr=0.001):
    """Train LSTM model."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    # Convert to tensors
    X_seq_t = torch.FloatTensor(X_seq_train).to(device)
    X_matchup_t = torch.FloatTensor(X_matchup_train).to(device)
    y_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    
    X_seq_v = torch.FloatTensor(X_seq_val).to(device)
    X_matchup_v = torch.FloatTensor(X_matchup_val).to(device)
    y_v = torch.FloatTensor(y_val).unsqueeze(1).to(device)
    
    model.to(device)
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        
        # Mini-batch training
        indices = np.random.permutation(len(X_seq_train))
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            
            optimizer.zero_grad()
            output = model(X_seq_t[batch_idx], X_matchup_t[batch_idx])
            loss = criterion(output, y_t[batch_idx])
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        train_losses.append(epoch_loss / n_batches)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_output = model(X_seq_v, X_matchup_v)
            val_loss = criterion(val_output, y_v).item()
            val_losses.append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Loss={val_losses[-1]:.4f}")
    
    return train_losses, val_losses

In [ ]:
def evaluate_lstm(model, X_seq, X_matchup, y_true):
    """Evaluate LSTM model."""
    model.eval()
    with torch.no_grad():
        X_seq_t = torch.FloatTensor(X_seq).to(device)
        X_matchup_t = torch.FloatTensor(X_matchup).to(device)
        y_pred = model(X_seq_t, X_matchup_t).cpu().numpy().flatten()
    
    metrics = EvaluationMetrics(y_true, y_pred)
    return {
        'auc_roc': metrics.auc_roc(),
        'brier_score': metrics.brier_score(),
        'accuracy': metrics.accuracy(),
    }, y_pred

## 3. Cross-Validation Training

In [ ]:
# Train with default parameters
fold_results = []
all_attention_weights = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splitter.split(df_clean)):
    print(f"\n{'='*50}")
    print(f"Fold {fold_idx + 1}")
    print(f"{'='*50}")
    print(f"Train: {len(train_idx):,} samples, Val: {len(val_idx):,} samples")
    
    # Split data
    X_seq_train = X_seq[train_idx]
    X_seq_val = X_seq[val_idx]
    X_matchup_train = X_matchup[train_idx]
    X_matchup_val = X_matchup[val_idx]
    y_train = y[train_idx]
    y_val = y[val_idx]
    
    # Scale
    X_seq_train_scaled = scaler_seq.fit_transform(X_seq_train.reshape(-1, n_rolling_features)).reshape(X_seq_train.shape)
    X_seq_val_scaled = scaler_seq.transform(X_seq_val.reshape(-1, n_rolling_features)).reshape(X_seq_val.shape)
    X_matchup_train_scaled = scaler_matchup.fit_transform(X_matchup_train)
    X_matchup_val_scaled = scaler_matchup.transform(X_matchup_val)
    
    # Initialize model
    model = UpsetLSTM(
        sequence_features=n_rolling_features,
        matchup_features=n_matchup_features,
        hidden_size=64,
        num_layers=2,
        dropout=0.3
    )
    
    # Train
    train_losses, val_losses = train_lstm(
        model, 
        X_seq_train_scaled, X_matchup_train_scaled, y_train,
        X_seq_val_scaled, X_matchup_val_scaled, y_val,
        epochs=100, batch_size=64, lr=0.001
    )
    
    # Evaluate
    metrics, y_pred = evaluate_lstm(model, X_seq_val_scaled, X_matchup_val_scaled, y_val)
    fold_results.append(metrics)
    
    print(f"\nFold {fold_idx + 1} Results:")
    print(f"  AUC-ROC: {metrics['auc_roc']:.4f}")
    print(f"  Brier Score: {metrics['brier_score']:.4f}")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")

In [ ]:
# Summary of CV results
cv_df = pd.DataFrame(fold_results)
cv_df.index = [f'Fold {i+1}' for i in range(len(cv_df))]
print("\nCross-Validation Results:")
print(cv_df)
print(f"\nMean AUC-ROC: {cv_df['auc_roc'].mean():.4f} (+/- {cv_df['auc_roc'].std():.4f})")
print(f"Mean Brier Score: {cv_df['brier_score'].mean():.4f} (+/- {cv_df['brier_score'].std():.4f})")

## 4. Hyperparameter Tuning with Optuna

In [ ]:
def optuna_objective(trial):
    """Optuna objective for LSTM hyperparameter tuning."""
    # Hyperparameters to tune
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 0.0001, 0.01, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    
    cv_scores = []
    
    for train_idx, val_idx in cv_splitter.split(df_clean):
        # Split and scale
        X_seq_train = X_seq[train_idx]
        X_seq_val = X_seq[val_idx]
        X_matchup_train = X_matchup[train_idx]
        X_matchup_val = X_matchup[val_idx]
        y_train = y[train_idx]
        y_val = y[val_idx]
        
        scaler_seq_local = StandardScaler()
        scaler_matchup_local = StandardScaler()
        
        X_seq_train_scaled = scaler_seq_local.fit_transform(X_seq_train.reshape(-1, n_rolling_features)).reshape(X_seq_train.shape)
        X_seq_val_scaled = scaler_seq_local.transform(X_seq_val.reshape(-1, n_rolling_features)).reshape(X_seq_val.shape)
        X_matchup_train_scaled = scaler_matchup_local.fit_transform(X_matchup_train)
        X_matchup_val_scaled = scaler_matchup_local.transform(X_matchup_val)
        
        # Train model
        model = UpsetLSTM(
            sequence_features=n_rolling_features,
            matchup_features=n_matchup_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        )
        
        # Quiet training
        train_lstm(
            model,
            X_seq_train_scaled, X_matchup_train_scaled, y_train,
            X_seq_val_scaled, X_matchup_val_scaled, y_val,
            epochs=30, batch_size=batch_size, lr=lr
        )
        
        # Evaluate
        metrics, _ = evaluate_lstm(model, X_seq_val_scaled, X_matchup_val_scaled, y_val)
        cv_scores.append(metrics['auc_roc'])
    
    return np.mean(cv_scores)

# Run optimization (reduced trials for demo)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(optuna_objective, n_trials=20, show_progress_bar=True)

print(f"\nBest AUC-ROC: {study.best_value:.4f}")
print(f"Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

## 5. Train Final Model and Analyze Attention

In [ ]:
# Train final model with best parameters on all data
# Scale all data
X_seq_scaled = scaler_seq.fit_transform(X_seq.reshape(-1, n_rolling_features)).reshape(X_seq.shape)
X_matchup_scaled = scaler_matchup.fit_transform(X_matchup)

final_model = UpsetLSTM(
    sequence_features=n_rolling_features,
    matchup_features=n_matchup_features,
    **{k: v for k, v in study.best_params.items() if k in ['hidden_size', 'num_layers', 'dropout']}
)

# Use a portion for validation to monitor training
train_size = int(0.8 * len(X_seq_scaled))
train_losses, val_losses = train_lstm(
    final_model,
    X_seq_scaled[:train_size], X_matchup_scaled[:train_size], y[:train_size],
    X_seq_scaled[train_size:], X_matchup_scaled[train_size:], y[train_size:],
    epochs=100, batch_size=study.best_params.get('batch_size', 64),
    lr=study.best_params.get('lr', 0.001)
)

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('LSTM Training History')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/lstm_training_history.png', dpi=150)
plt.show()

In [ ]:
# Get attention weights
final_model.eval()
with torch.no_grad():
    X_seq_t = torch.FloatTensor(X_seq_scaled).to(device)
    X_matchup_t = torch.FloatTensor(X_matchup_scaled).to(device)
    
    # Forward pass through LSTM
    lstm_out, _ = final_model.lstm(X_seq_t)
    
    # Get attention weights
    attn_weights = torch.softmax(final_model.attention(lstm_out), dim=1)
    attn_weights_np = attn_weights.cpu().numpy()

print(f"Attention weights shape: {attn_weights_np.shape}")
print(f"Mean attention per time step: {attn_weights_np.mean(axis=0).flatten()}")

In [ ]:
# Visualize attention pattern
fig, ax = plt.subplots(figsize=(8, 5))
mean_attn = attn_weights_np.mean(axis=0).flatten()
ax.bar(range(len(mean_attn)), mean_attn, color='steelblue', alpha=0.7)
ax.set_xlabel('Sequence Position')
ax.set_ylabel('Mean Attention Weight')
ax.set_title('LSTM Attention Weights by Sequence Position')
ax.set_xticks(range(len(mean_attn)))
ax.set_xticklabels([f't-{len(mean_attn)-i}' for i in range(len(mean_attn))])
plt.tight_layout()
plt.savefig('../results/figures/lstm_attention_weights.png', dpi=150)
plt.show()

## 6. Save Model

In [ ]:
# Save model and metadata
import os
os.makedirs('../results/models', exist_ok=True)

model_path = '../results/models/lstm_upset_model.pt'
torch.save({
    'model_state_dict': final_model.state_dict(),
    'model_config': {
        'sequence_features': n_rolling_features,
        'matchup_features': n_matchup_features,
        **{k: v for k, v in study.best_params.items() if k in ['hidden_size', 'num_layers', 'dropout']}
    },
    'best_params': study.best_params,
    'cv_auc': study.best_value,
    'scaler_seq_mean': scaler_seq.mean_,
    'scaler_seq_scale': scaler_seq.scale_,
    'scaler_matchup_mean': scaler_matchup.mean_,
    'scaler_matchup_scale': scaler_matchup.scale_,
    'rolling_features': rolling_features,
    'matchup_features': matchup_features,
}, model_path)

print(f"Model saved to {model_path}")

In [ ]:
# Final summary
print("=" * 50)
print("LSTM MODEL SUMMARY")
print("=" * 50)
print(f"\nCross-validation AUC-ROC: {study.best_value:.4f}")
print(f"\nBest architecture:")
print(f"  Hidden size: {study.best_params.get('hidden_size', 64)}")
print(f"  Num layers: {study.best_params.get('num_layers', 2)}")
print(f"  Dropout: {study.best_params.get('dropout', 0.3):.2f}")
print(f"\nModel saved to: results/models/lstm_upset_model.pt")